# 3. LoRA 微调 OpenVLA-UAV

本 Notebook 展示如何使用 LoRA (Low-Rank Adaptation) 对 OpenVLA 基座模型进行无人机动作微调。

**学习目标：**
- 理解 LoRA 微调的原理和优势
- 构建 UAV 数据集和 DataLoader
- 配置和应用 LoRA adapter
- 运行训练循环并监控指标
- 保存微调 checkpoint

## 3.1 LoRA 原理简介

全量微调 7B 模型需要巨大的显存和算力。**LoRA** 通过在模型权重矩阵旁添加低秩分解矩阵来实现高效微调：

```
原始权重:  W (d x d)
LoRA 分解: W' = W + B @ A   其中 A (r x d), B (d x r), r << d

例: d=4096, r=32
  原始参数: 4096 x 4096 = 16.7M
  LoRA 参数: 4096 x 32 + 32 x 4096 = 0.26M  (仅 1.6%)
```

优势：
- 仅训练 ~1% 的参数，大幅减少显存
- 训练速度快，收敛也快
- 推理时可将 LoRA 权重合并回原模型，无额外开销

## 3.2 环境准备

In [1]:
import sys
import os
import json
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'disabled'

PROJECT_ROOT = "/root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV"
sys.path.insert(0, PROJECT_ROOT)

MODEL_PATH = "/root/autodl-fs/claude/models/models--openvla--openvla-7b/snapshots/47a0ec7fc4ec123775a391911046cf33cf9ed83f"
DATA_DIR = "/root/autodl-fs/claude/data/uav_flow_subset"
RUN_DIR = "/root/autodl-fs/claude/runs/notebook_demo"
ADAPTER_DIR = "/root/autodl-fs/claude/adapter-tmp/notebook_demo"

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA GeForce RTX 3080
显存: 21.0 GB


## 3.3 注册模型 & 加载 Processor

In [2]:
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticProcessor, PrismaticImageProcessor
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor, AutoModelForVision2Seq

AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"Processor loaded. Vocab size: {processor.tokenizer.vocab_size}")

Processor loaded. Vocab size: 32000


## 3.4 构建数据集

`SimpleVLADataset` 是 UAV-Flow 专用的数据集类，它：
1. 从文件夹加载轨迹数据
2. 将全局坐标转换为局部坐标系下的相对动作
3. 计算动作统计量 (用于归一化)
4. 将图像 + 指令 + 动作打包为模型可接受的格式

In [3]:
from prismatic.models.backbones.llm.prompting import PurePromptBuilder
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.vla.datasets.uav_dataset import SimpleVLADataset

action_tokenizer = ActionTokenizer(processor.tokenizer)

print("加载数据集...")
dataset = SimpleVLADataset(
    data_path=DATA_DIR,
    image_transform=processor.image_processor,
    tokenizer=processor.tokenizer,
    action_tokenizer=action_tokenizer,
    prompt_builder_fn=PurePromptBuilder,
    is_train=True,
)

print(f"\n数据集加载完成!")
print(f"样本总数: {len(dataset)}")
print(f"动作统计:")
print(f"  mean: {dataset.action_mean}")
print(f"  std:  {dataset.action_std}")
print(f"  min (1%): {dataset.action_min}")
print(f"  max (99%): {dataset.action_max}")

加载数据集...


04/16 [22:35:22] INFO     | >> Action statistics:                                                ]8;id=828562;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py\uav_dataset.py]8;;\:]8;id=828563;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py#207\207]8;;\

                 INFO     | >> mean: [ 9.76162286e-02  1.13318791e-02  9.67774008e-05            ]8;id=828569;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py\uav_dataset.py]8;;\:]8;id=828570;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py#208\208]8;;\
                          -6.17306328e-03]                                                                         

                 INFO     | >> std: [0.1443502  0.0833911  0.02257356 0.12307155]                ]8;id=828576;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py\uav_dataset.py]8;;\:]8;id=828577;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py#209\209]8;;\

                 INFO     | >> 1st percentile (min approx): [-0.24067513 -0.23219142 -0.08960007 ]8;id=828583;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py\uav_dataset.py]8;;\:]8;id=828584;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py#210\210]8;;\
                          -0.13755473]                                                                             

                 INFO     | >> 99th percentile (max approx): [0.55850679 0.23932622 0.09363774   ]8;id=828590;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py\uav_dataset.py]8;;\:]8;id=828591;file:///root/autodl-fs/claude/UAV-Flow/OpenVLA-UAV/prismatic/vla/datasets/uav_dataset.py#211\211]8;;\
                          0.10081557]                                                                              


数据集加载完成!
样本总数: 39518
动作统计:
  mean: [ 9.76162286e-02  1.13318791e-02  9.67774008e-05 -6.17306328e-03]
  std:  [0.1443502  0.0833911  0.02257356 0.12307155]
  min (1%): [-0.24067513 -0.23219142 -0.08960007 -0.13755473]
  max (99%): [0.55850679 0.23932622 0.09363774 0.10081557]


In [4]:
# 查看一个样本
sample_iter = iter(dataset)
sample = next(sample_iter)

print("单个训练样本:")
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
    else:
        print(f"  {key}: {val}")

# 解码看看 label 中的动作 token
action_mask = sample['labels'] > action_tokenizer.action_token_begin_idx
action_ids = sample['labels'][action_mask].numpy()
print(f"\n动作 token 数: {len(action_ids)}")
if len(action_ids) > 0:
    decoded = action_tokenizer.decode_token_ids_to_actions(action_ids)
    print(f"解码后动作 [-1,1]: {decoded}")

单个训练样本:
  pixel_values: shape=torch.Size([6, 224, 224]), dtype=torch.float32
  input_ids: shape=torch.Size([49]), dtype=torch.int64
  labels: shape=torch.Size([49]), dtype=torch.int64
  attention_mask: shape=torch.Size([49]), dtype=torch.int64
  dataset_name: uav

动作 token 数: 4
解码后动作 [-1,1]: [-0.10980392  0.         -0.02352941  0.18039216]


## 3.5 加载模型 & 配置 LoRA

In [5]:
from peft import LoraConfig, get_peft_model

# 加载基座模型
print("加载 OpenVLA-7B 基座模型...")
vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda:0")

print(f"基座模型参数量: {sum(p.numel() for p in vla.parameters()) / 1e9:.2f}B")
print(f"显存占用: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

加载 OpenVLA-7B 基座模型...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

基座模型参数量: 7.54B
显存占用: 15.1 GB


In [6]:
# 配置 LoRA
lora_config = LoraConfig(
    r=32,                          # LoRA 秩 (越大越有表达力, 但参数越多)
    lora_alpha=16,                 # LoRA 缩放系数
    lora_dropout=0.0,              # Dropout 比例
    target_modules="all-linear",   # 对所有线性层添加 LoRA
    init_lora_weights="gaussian",  # 初始化方式
)

print("LoRA 配置:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  target_modules: {lora_config.target_modules}")

# 应用 LoRA
vla = get_peft_model(vla, lora_config)
vla.print_trainable_parameters()

LoRA 配置:
  rank (r): 32
  alpha: 16
  target_modules: all-linear
trainable params: 110,828,288 || all params: 7,652,065,472 || trainable%: 1.4483


可以看到，LoRA 只训练约 **1%** 的参数 (~70M out of 7B)，大幅减少了显存需求。

## 3.6 构建 DataLoader

In [7]:
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from torch.utils.data import DataLoader

# Collator: 将不同长度的样本 padding 到统一长度
collator = PaddedCollatorForActionPrediction(
    model_max_length=processor.tokenizer.model_max_length,
    pad_token_id=processor.tokenizer.pad_token_id,
    padding_side="right",
)

# DataLoader
BATCH_SIZE = 4  # RTX 4080 SUPER 32GB 可用; OOM 则改为 2

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collator,
    drop_last=True,
    num_workers=2,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"DataLoader created.")

# 验证一个 batch
batch = next(iter(dataloader))
print(f"\nBatch 内容:")
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")

Batch size: 4
DataLoader created.

Batch 内容:
  pixel_values: shape=torch.Size([4, 6, 224, 224]), dtype=torch.float32
  input_ids: shape=torch.Size([4, 53]), dtype=torch.int64
  attention_mask: shape=torch.Size([4, 53]), dtype=torch.bool
  labels: shape=torch.Size([4, 53]), dtype=torch.int64


## 3.7 训练循环

这里我们训练 **50 步** 作为演示（完整训练需要 500-5000 步）。

In [8]:
from torch.optim import AdamW
from collections import deque

# 训练配置
MAX_STEPS = 50
LEARNING_RATE = 5e-4
GRAD_ACCUMULATION = 2  # 有效 batch = 4 * 2 = 8

# 优化器
trainable_params = [p for p in vla.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE)

# 记录指标
history = {'step': [], 'loss': [], 'accuracy': [], 'l1_loss': []}

print(f"开始训练:")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUMULATION} = {BATCH_SIZE * GRAD_ACCUMULATION}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  显存: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("-" * 60)

开始训练:
  Max steps: 50
  Batch size: 4 x 2 = 8
  Learning rate: 0.0005
  显存: 15.4 GB
------------------------------------------------------------


In [9]:
vla.train()
optimizer.zero_grad()
device_id = 0

for batch_idx, batch in enumerate(dataloader):
    # Forward pass
    with torch.autocast("cuda", dtype=torch.bfloat16):
        output = vla(
            input_ids=batch["input_ids"].to(device_id),
            attention_mask=batch["attention_mask"].to(device_id),
            pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
            labels=batch["labels"],
        )
        loss = output.loss

    # Backward
    normalized_loss = loss / GRAD_ACCUMULATION
    normalized_loss.backward()

    # 计算动作准确率
    action_logits = output.logits[:, vla.vision_backbone.featurizer.patch_embed.num_patches : -1]
    action_preds = action_logits.argmax(dim=2)
    action_gt = batch["labels"][:, 1:].to(action_preds.device)
    mask = action_gt > action_tokenizer.action_token_begin_idx
    correct_preds = (action_preds == action_gt) & mask
    accuracy = correct_preds.sum().float() / mask.sum().float() if mask.sum() > 0 else torch.tensor(0.0)

    # L1 loss
    try:
        pred_actions = torch.tensor(action_tokenizer.decode_token_ids_to_actions(action_preds[mask].cpu().numpy()))
        gt_actions = torch.tensor(action_tokenizer.decode_token_ids_to_actions(action_gt[mask].cpu().numpy()))
        l1_loss = torch.nn.functional.l1_loss(pred_actions, gt_actions).item()
    except:
        l1_loss = 0.0

    # Optimizer step
    gradient_step = batch_idx // GRAD_ACCUMULATION
    if (batch_idx + 1) % GRAD_ACCUMULATION == 0:
        optimizer.step()
        optimizer.zero_grad()

        # 记录
        history['step'].append(gradient_step)
        history['loss'].append(loss.item())
        history['accuracy'].append(accuracy.item())
        history['l1_loss'].append(l1_loss)

        if gradient_step % 5 == 0:
            print(f"  Step {gradient_step:>3d} | loss={loss.item():.4f} | acc={accuracy.item():.3f} | l1={l1_loss:.4f}")

    if gradient_step >= MAX_STEPS:
        break

print("-" * 60)
print(f"训练完成! 共 {MAX_STEPS} 步")
print(f"最终 loss: {history['loss'][-1]:.4f}")
print(f"最终 accuracy: {history['accuracy'][-1]:.3f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 19.58 GiB of which 15.19 MiB is free. Process 14127 has 2.75 GiB memory in use. Process 14127 has 2.75 GiB memory in use. Process 57727 has 1.70 GiB memory in use. Including non-PyTorch memory, this process has 15.02 GiB memory in use. Of the allocated memory 14.68 GiB is allocated by PyTorch, and 77.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 3.8 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['step'], history['loss'], 'b-')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['step'], history['accuracy'], 'g-')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Action Token Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['step'], history['l1_loss'], 'r-')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('L1 Loss')
axes[2].set_title('Action L1 Loss')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Training Metrics ({MAX_STEPS} steps, LoRA r=32)', fontsize=14)
plt.tight_layout()
plt.show()

## 3.9 保存 Checkpoint

保存 LoRA adapter 和合并后的完整模型。

In [ ]:
from prismatic.vla.datasets.uav_dataset import SimpleVLADataset

# 保存 LoRA adapter
print("保存 LoRA adapter...")
processor.save_pretrained(RUN_DIR)
vla.save_pretrained(ADAPTER_DIR)

# 保存动作统计量 (推理时需要)
norm_stats = {
    "sim": {
        "action": {
            "mean": dataset.action_mean.tolist(),
            "std": dataset.action_std.tolist(),
            "min": dataset.action_min.tolist(),
            "max": dataset.action_max.tolist(),
            "q01": dataset.action_min.tolist(),
            "q99": dataset.action_max.tolist(),
        }
    }
}
with open(os.path.join(RUN_DIR, "dataset_statistics.json"), 'w') as f:
    json.dump(norm_stats, f, indent=2)

print(f"Adapter 保存到: {ADAPTER_DIR}")
print(f"Processor 保存到: {RUN_DIR}")
print(f"统计量保存到: {RUN_DIR}/dataset_statistics.json")

In [ ]:
# (可选) 合并 LoRA 权重到基座模型
from peft import PeftModel

print("合并 LoRA 权重到基座模型...")
base_vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
)
merged_vla = PeftModel.from_pretrained(base_vla, ADAPTER_DIR)
merged_vla = merged_vla.merge_and_unload()
merged_vla.save_pretrained(RUN_DIR)

print(f"合并后的模型保存到: {RUN_DIR}")
print(f"目录内容: {os.listdir(RUN_DIR)}")

In [ ]:
# 清理显存
del vla, base_vla, merged_vla
torch.cuda.empty_cache()
print("显存已释放.")

## 小结

- **LoRA** 只训练 ~1% 参数，32GB GPU 即可微调 7B 模型
- 训练数据流: 图像 + 指令 → Processor → 模型前向 → 动作 token 交叉熵损失
- 关键指标: **loss** (越低越好), **action accuracy** (token 匹配率), **L1 loss** (连续动作误差)
- 50 步只是演示, 实际训练需 500-5000 步 (对应 `bash run_finetune.sh`)
- 保存时要同时保存 **模型权重** 和 **动作统计量** (dataset_statistics.json)

**下一步:** 在 Notebook 04 中，我们将加载微调后的模型进行推理预测。